In [ ]:
import time
from tqdm import tqdm
import logging
import sys
import config
from utils.data_process_helper import extract_card_data, download_card_images

In [7]:
logging.basicConfig(
    level="INFO",
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

In [4]:
json_files = list(config.TRANING_SOURCE_PATH.glob("*.json"))
df = extract_card_data(json_files)
df.count()

100%|██████████| 22/22 [00:00<00:00, 35.72it/s]


layout      11907
name        11907
setCode     11907
number      11907
isOldSet    11907
dtype: int64

In [ ]:
download_card_images(df, config.TRAINING_IMAGE_PATH)

2025-10-04 03:50:19 - INFO - Total cards: 11596
2025-10-04 03:50:19 - INFO - Already downloaded: 11596
2025-10-04 03:50:19 - INFO - To download: 0


In [ ]:
df.to_parquet(config.TRAINING_DATA_FILE_PATH, engine='pyarrow', compression='snappy', index=False)
df.describe()

,layout,name,setCode,number,isOldSet
count,11907,11907,11907,11907,11907
unique,13,6226,22,2428,2
top,normal,Mountain,SLD,23,False
freq,11130,93,2241,23,10558


In [99]:
from utils.data_process_helper import load_parquet_data
from typing import Dict, List
from pathlib import Path
import yaml
import random
import shutil
import cv2
from sklearn.model_selection import train_test_split

In [100]:
data = load_parquet_data(config.TRAINING_DATA_FILE_PATH, config.TRAINING_IMAGE_PATH)

2025-10-05 01:31:14 - INFO - Loading parquet file: _data/training/training_data.parquet
2025-10-05 01:31:14 - INFO - Loaded 11907 rows
2025-10-05 01:31:14 - INFO - Columns: ['layout', 'name', 'setCode', 'number', 'isOldSet']
2025-10-05 01:31:15 - INFO - Created 11596 annotations
2025-10-05 01:31:15 - INFO - ✗ Missing 0 images


In [101]:
MODERN_CARD_REGIONS = {
    "card_name": {"x": 0.50, "y": 0.08, "w": 0.85, "h": 0.06},
    "set_code": {"x": 0.08, "y": 0.965, "w": 0.08, "h": 0.025},
    "number": {"x": 0.11, "y": 0.945, "w": 0.18, "h": 0.025},
    "language": {"x": 0.16, "y": 0.965, "w": 0.06, "h": 0.025},
}

OLD_CARD_REGIONS = {
    "card_name": {"x": 0.50, "y": 0.06, "w": 0.92, "h": 0.06},
}

# Class mapping
CLASS_NAMES = {
    "card_name": 0,
    "set_code": 1,
    "number": 2,
    "language": 3,
}

In [102]:
def generate_yolo_label(is_old_set: bool) -> List[str]:
    """
    Generate YOLO format labels based on card type.
    
    Args:
        is_old_set: True if old frame card, False if modern
        
    Returns:
        List of YOLO format label lines
    """
    labels = []
    
    if is_old_set:
        regions = OLD_CARD_REGIONS
    else:
        regions = MODERN_CARD_REGIONS
    
    for region_name, coords in regions.items():
        class_id = CLASS_NAMES[region_name]
        label_line = f"{class_id} {coords['x']:.6f} {coords['y']:.6f} {coords['w']:.6f} {coords['h']:.6f}"
        labels.append(label_line)
    
    return labels

In [103]:
def visualize_labels(
    image_path: Path,
    label_lines: List[str],
    output_path: Path,
) -> None:
    """
    Draw bounding boxes on image to visualize YOLO labels.
    
    Args:
        image_path: Path to card image
        label_lines: YOLO format label lines
        output_path: Path to save visualization
    """
    class_names = {v: k for k, v in CLASS_NAMES.items()}
    
    img = cv2.imread(str(image_path))
    if img is None:
        logger.warning(f"Could not read image: {image_path}")
        return
    
    h, w = img.shape[:2]
    
    colors = {
        0: (0, 255, 0),      # card_name - green
        1: (255, 0, 0),      # set_code - blue
        2: (0, 0, 255),      # collector_number - red
        3: (255, 255, 0),    # language_indicator - cyan
    }
    
    for line in label_lines:
        parts = line.strip().split()
        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:5])
        
        x_center_px = int(x_center * w)
        y_center_px = int(y_center * h)
        width_px = int(width * w)
        height_px = int(height * h)
        
        x1 = int(x_center_px - width_px / 2)
        y1 = int(y_center_px - height_px / 2)
        x2 = int(x_center_px + width_px / 2)
        y2 = int(y_center_px + height_px / 2)
        
        color = colors.get(class_id, (255, 255, 255))
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        
        label_text = class_names.get(class_id, f"Class {class_id}")
        cv2.putText(img, label_text, (x1, y1 - 5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    cv2.imwrite(str(output_path), img)

In [104]:
def create_visualization_samples(
    image_path: Path,
    output_path: Path,
    data: Dict,
    num_samples: int = 10
) -> None:
    """
    Create visualization samples for both old and new card formats.
    
    Args:
        image_path: Path to card images
        output_path: Path to save visualizations
        data: Loaded card data
        num_samples: Number of samples per format to visualize
    """
    viz_dir = output_path / "visualizations"
    viz_dir.mkdir(parents=True, exist_ok=True)
    
    old_cards = [k for k, v in data.items() if v.isOldSet]
    new_cards = [k for k, v in data.items() if not v.isOldSet]
    
    old_samples = random.sample(old_cards, min(num_samples, len(old_cards)))
    new_samples = random.sample(new_cards, min(num_samples, len(new_cards)))
    
    logger.info(f"Creating {len(old_samples)} old format card visualizations...")
    for filename in tqdm(old_samples):
        img_path = image_path / filename
        if img_path.exists():
            labels = generate_yolo_label(True)
            viz_path = viz_dir / f"old_{filename}"
            visualize_labels(img_path, labels, viz_path)
    
    logger.info(f"Creating {len(new_samples)} new format card visualizations...")
    for filename in tqdm(new_samples):
        img_path = image_path / filename
        if img_path.exists():
            labels = generate_yolo_label(False)
            viz_path = viz_dir / f"new_{filename}"
            visualize_labels(img_path, labels, viz_path)
    
    logger.info(f"Visualizations saved to: {viz_dir}")
    logger.info("Check these images to verify bounding box positions!")
    logger.info("Colors: Green=card_name, Blue=set_code, Red=collector_number, Cyan=language")

In [ ]:
shutil.rmtree(config.REGION_DETECTION_PATH / "visualizations")
create_visualization_samples(config.TRAINING_IMAGE_PATH, config.REGION_DETECTION_PATH, data, 100)

2025-10-05 01:10:24 - INFO - Creating 100 old format card visualizations...


100%|██████████| 100/100 [00:00<00:00, 220.93it/s]

2025-10-05 01:10:24 - INFO - Creating 100 new format card visualizations...



100%|██████████| 100/100 [00:00<00:00, 254.79it/s]

2025-10-05 01:10:25 - INFO - Visualizations saved to: _data/training/region_detection/visualizations
2025-10-05 01:10:25 - INFO - Check these images to verify bounding box positions!
2025-10-05 01:10:25 - INFO - Colors: Green=card_name, Blue=set_code, Red=collector_number, Cyan=language


In [ ]:
def create_yolo_dataset(
    data_path: Path,
    image_path: Path,
    output_path: Path,
    test_size: float = 0.2,
) -> None:
    """Create full YOLO dataset with train/val split."""
    
    logger.info("=" * 60)
    logger.info("GENERATING YOLO DATASET")
    logger.info("=" * 60)
    
    data = load_parquet_data(data_path, image_path)
    keys = list(data.keys())
    labels = [data[key].isOldSet for key in keys]
    
    logger.info(f"Total images: {len(keys)}")
    logger.info(f"Old frames: {sum(labels)}")
    logger.info(f"New frames: {len(labels) - sum(labels)}")
    
    train_files, val_files = train_test_split(
        keys, test_size=test_size, stratify=labels, random_state=42
    )
    
    logger.info(f"Training: {len(train_files)}, Validation: {len(val_files)}")
    
    train_img_dir = output_path / "train" / "images"
    train_lbl_dir = output_path / "train" / "labels"
    val_img_dir = output_path / "val" / "images"
    val_lbl_dir = output_path / "val" / "labels"
    
    for dir_path in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir]:
        dir_path.mkdir(parents=True, exist_ok=True)
    
    logger.info("Processing training data...")
    for filename in tqdm(train_files):
        is_old = data[filename].isOldSet
        
        src_img = image_path / filename
        dst_img = train_img_dir / filename
        if src_img.exists():
            shutil.copy(src_img, dst_img)
        
        yolo_labels = generate_yolo_label(is_old)
        label_filename = Path(filename).stem + ".txt"
        label_path = train_lbl_dir / label_filename
        
        with open(label_path, 'w') as f:
            f.write('\n'.join(yolo_labels))
    
    logger.info("Processing validation data...")
    for filename in tqdm(val_files):
        is_old = data[filename].isOldSet
        
        src_img = image_path / filename
        dst_img = val_img_dir / filename
        if src_img.exists():
            shutil.copy(src_img, dst_img)
        
        yolo_labels = generate_yolo_label(is_old)
        label_filename = Path(filename).stem + ".txt"
        label_path = val_lbl_dir / label_filename
        
        with open(label_path, 'w') as f:
            f.write('\n'.join(yolo_labels))
    
    data_yaml = {
        'path': str(output_path.absolute()),
        'train': 'train/images',
        'val': 'val/images',
        'names': {v: k for k, v in CLASS_NAMES.items()}
    }
    
    yaml_path = output_path / "data.yaml"
    with open(yaml_path, 'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False)
    
    logger.info("=" * 60)
    logger.info("YOLO Dataset Created Successfully!")
    logger.info(f"Dataset location: {output_path}")
    logger.info(f"Config file: {yaml_path}")
    logger.info(f"Train images: {len(list(train_img_dir.glob('*')))}")
    logger.info(f"Train labels: {len(list(train_lbl_dir.glob('*.txt')))}")
    logger.info(f"Val images: {len(list(val_img_dir.glob('*')))}")
    logger.info(f"Val labels: {len(list(val_lbl_dir.glob('*.txt')))}")
    logger.info("=" * 60)

In [ ]:
create_yolo_dataset(
    data_path=config.TRAINING_DATA_FILE_PATH,
    image_path=config.TRAINING_IMAGE_PATH,
    output_path=config.REGION_DETECTION_PATH,
    test_size=0.2,
)

2025-10-05 01:33:36 - INFO - ============================================================
2025-10-05 01:33:36 - INFO - GENERATING YOLO DATASET
2025-10-05 01:33:36 - INFO - ============================================================
2025-10-05 01:33:36 - INFO - Loading parquet file: _data/training/training_data.parquet
2025-10-05 01:33:36 - INFO - Loaded 11907 rows
2025-10-05 01:33:36 - INFO - Columns: ['layout', 'name', 'setCode', 'number', 'isOldSet']
2025-10-05 01:33:36 - INFO - Created 11596 annotations
2025-10-05 01:33:36 - INFO - ✗ Missing 0 images
2025-10-05 01:33:36 - INFO - Total images: 11596
2025-10-05 01:33:36 - INFO - Old frames: 1349
2025-10-05 01:33:36 - INFO - New frames: 10247
2025-10-05 01:33:36 - INFO - Training: 9276, Validation: 2320
2025-10-05 01:33:36 - INFO - Processing training data...
2025-10-05 01:33:38 - INFO - Processing validation data...
2025-10-05 01:33:39 - INFO - ============================================================
2025-10-05 01:33:39 - INFO - 